# Pathwise Delta for the American Put

This notebook presents the first pathwise delta study for the American put baseline. The objective is not just to produce a delta estimate, but to show how the estimate behaves relative to finite-difference LSMC and to external validation benchmarks.


## Method Summary

We reuse the LSMC stopping rule from the pricing run and differentiate the discounted intrinsic payoff path by path while **holding the stopping policy fixed**. Under GBM,

$$\frac{\partial S_t}{\partial S_0} = \frac{S_t}{S_0},$$

so the pathwise delta sample at the realized exercise time $\tau$ is

$$e^{-r\tau}\left(-\frac{S_{\tau}}{S_0}\right) \mathbf{1}_{\{K-S_{\tau}>0\}}.$$


## Theory Note: Why This Is a Fixed-Policy Approximation

An American option value is the solution of an **optimal stopping** problem, not just the expectation of a payoff under a fixed exercise date. In the current implementation, the pathwise estimator differentiates the payoff under the **learned LSMC stopping rule** and keeps that stopping rule fixed during differentiation.

That is why benchmark agreement is important here: it shows the estimator is useful in the current setup, but it does **not** by itself prove that the code is computing the exact derivative of the true optimal-stopping value. In the report, this method should therefore be described as a validated first-order pathwise estimator under a fixed-policy assumption.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
src_root = project_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

figure_dir = project_root / "assets" / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)
plt.style.use("seaborn-v0_8-whitegrid")

from lsmc_greeks.benchmarks.binomial import american_put_delta_binomial
from lsmc_greeks.benchmarks.finite_difference import american_put_delta_finite_difference
from lsmc_greeks.greeks.finite_diff import estimate_delta_fd
from lsmc_greeks.greeks.pathwise import estimate_delta_pathwise
from lsmc_greeks.pricer import LSMCConfig, lsm_american_put

# Notebook 03 uses fixed seed values (42 for single-seed cells, [11..89] for repeated).
# Notebook 06 uses np.random.SeedSequence(ROOT_SEED=203) for reproducibility.
# The two conventions are independent; results are comparable across notebooks
# because both use the same baseline parameters and basis_degree=3.

def save_figure(fig, filename):
    path = figure_dir / filename
    fig.savefig(path, dpi=220, bbox_inches="tight")
    return path


## Benchmark Setup

The benchmark case is the at-the-money American put. We compare the LSMC estimators against both a binomial benchmark and a finite-difference PDE benchmark.


In [ ]:
spot = 40.0
strike = 40.0
rate = 0.06
sigma = 0.20
maturity = 1.0

base_config = LSMCConfig(n_paths=40_000, n_steps_per_year=50, basis_degree=3, seed=42)
benchmark_binomial = american_put_delta_binomial(spot, strike, rate, sigma, maturity, n_steps=1000, bump=0.5)
benchmark_pde = american_put_delta_finite_difference(spot, strike, rate, sigma, maturity, n_space_steps=200, n_time_steps_per_year=2000, bump=0.5)
pd.DataFrame([
    {"benchmark": "Binomial", "delta": benchmark_binomial},
    {"benchmark": "Finite-difference PDE", "delta": benchmark_pde},
]).round(6)


## Baseline Delta Comparison

We first compare the two LSMC estimators at the base configuration. This is the headline comparison for the README: does the pathwise delta track the external benchmarks, and does it do so without the bump-size tuning required by bump-and-revalue?


In [ ]:
fd_lsmc = estimate_delta_fd(spot, strike, rate, sigma, maturity, config=base_config, bump=0.5, seed=42)
pathwise = estimate_delta_pathwise(spot, strike, rate, sigma, maturity, config=base_config, seed=42)

single_case = pd.DataFrame([
    {"method": "LSMC finite difference", "delta": fd_lsmc["estimate"], "std_error": fd_lsmc["std_error"], "runtime_sec": fd_lsmc["runtime_sec"], "abs_error_vs_binomial": abs(fd_lsmc["estimate"] - benchmark_binomial)},
    {"method": "LSMC pathwise", "delta": pathwise["estimate"], "std_error": pathwise["std_error"], "runtime_sec": pathwise["runtime_sec"], "abs_error_vs_binomial": abs(pathwise["estimate"] - benchmark_binomial)},
    {"method": "Binomial benchmark", "delta": benchmark_binomial, "std_error": None, "runtime_sec": None, "abs_error_vs_binomial": 0.0},
    {"method": "Finite-difference benchmark", "delta": benchmark_pde, "std_error": None, "runtime_sec": None, "abs_error_vs_binomial": abs(benchmark_pde - benchmark_binomial)},
]).round(6)
single_case


In [ ]:
plot_df = single_case.dropna(subset=["runtime_sec"]).copy()
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

axes[0].bar(plot_df["method"], plot_df["delta"], color=["#4c78a8", "#f58518"])
axes[0].axhline(benchmark_binomial, color="black", linestyle=":", linewidth=1.2, label="Binomial benchmark")
axes[0].axhline(benchmark_pde, color="#54a24b", linestyle="--", linewidth=1.2, label="PDE benchmark")
axes[0].set_title("Baseline delta estimates")
axes[0].set_ylabel("Delta")
axes[0].tick_params(axis="x", rotation=10)
axes[0].legend()

axes[1].bar(plot_df["method"], plot_df["abs_error_vs_binomial"], color=["#4c78a8", "#f58518"])
axes[1].set_title("Absolute error vs binomial")
axes[1].set_ylabel("Absolute error")
axes[1].tick_params(axis="x", rotation=10)

plt.tight_layout()
save_figure(fig, "pathwise_delta_baseline_comparison.png")
plt.show()


## Robustness 1: Repeated-Seed Path-Count Check

The earlier single-seed path-count profile is useful for intuition, but a stronger claim needs repeated runs. Here we repeat the experiment across several seeds and plot mean values with error bars. This does not make the result fully asymptotic, but it gives a more reliable picture than one seed alone.


In [ ]:
seeds = [11, 17, 23, 29, 35, 41, 47, 53, 59, 67, 71, 73, 79, 83, 89]
path_counts = [5_000, 10_000, 20_000, 40_000]
seed_rows = []

for n_paths in path_counts:
    for seed in seeds:
        config = LSMCConfig(n_paths=n_paths, n_steps_per_year=50, basis_degree=3, seed=seed)
        fd_est = estimate_delta_fd(spot, strike, rate, sigma, maturity, config=config, bump=0.5, seed=seed)
        pw_est = estimate_delta_pathwise(spot, strike, rate, sigma, maturity, config=config, seed=seed)
        seed_rows.append({
            "n_paths": n_paths,
            "seed": seed,
            "fd_delta": fd_est["estimate"],
            "fd_runtime_sec": fd_est["runtime_sec"],
            "fd_abs_error": abs(fd_est["estimate"] - benchmark_binomial),
            "pathwise_delta": pw_est["estimate"],
            "pathwise_runtime_sec": pw_est["runtime_sec"],
            "pathwise_abs_error": abs(pw_est["estimate"] - benchmark_binomial),
        })

seed_df = pd.DataFrame(seed_rows)
seed_df.head()


In [ ]:
seed_summary = seed_df.groupby("n_paths").agg(
    fd_delta_mean=("fd_delta", "mean"),
    fd_delta_std=("fd_delta", "std"),
    fd_abs_error_mean=("fd_abs_error", "mean"),
    fd_abs_error_std=("fd_abs_error", "std"),
    fd_runtime_mean=("fd_runtime_sec", "mean"),
    pathwise_delta_mean=("pathwise_delta", "mean"),
    pathwise_delta_std=("pathwise_delta", "std"),
    pathwise_abs_error_mean=("pathwise_abs_error", "mean"),
    pathwise_abs_error_std=("pathwise_abs_error", "std"),
    pathwise_runtime_mean=("pathwise_runtime_sec", "mean"),
).reset_index().round(6)
seed_summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

axes[0].errorbar(seed_summary["n_paths"], seed_summary["fd_delta_mean"], yerr=seed_summary["fd_delta_std"], marker="^", linewidth=2, capsize=4, label="LSMC finite difference")
axes[0].errorbar(seed_summary["n_paths"], seed_summary["pathwise_delta_mean"], yerr=seed_summary["pathwise_delta_std"], marker="d", linewidth=2, capsize=4, label="LSMC pathwise")
axes[0].axhline(benchmark_binomial, color="black", linestyle=":", linewidth=1.2, label="Binomial benchmark")
axes[0].set_title("Mean delta with seed-to-seed variation")
axes[0].set_xlabel("Number of paths")
axes[0].set_ylabel("Delta")
axes[0].legend()

axes[1].errorbar(seed_summary["n_paths"], seed_summary["fd_abs_error_mean"], yerr=seed_summary["fd_abs_error_std"], marker="^", linewidth=2, capsize=4, label="LSMC finite difference")
axes[1].errorbar(seed_summary["n_paths"], seed_summary["pathwise_abs_error_mean"], yerr=seed_summary["pathwise_abs_error_std"], marker="d", linewidth=2, capsize=4, label="LSMC pathwise")
axes[1].set_title("Mean absolute error with seed-to-seed variation")
axes[1].set_xlabel("Number of paths")
axes[1].set_ylabel("Absolute error vs binomial")
axes[1].legend()

plt.tight_layout()
save_figure(fig, "pathwise_delta_path_count.png")
plt.show()


## Robustness 2: Bump-Size Sensitivity

Bump-and-revalue depends on a user-chosen bump size. The pathwise estimator does not have this tuning parameter, so this sweep helps explain why the pathwise method is attractive in practice.


In [ ]:
bump_rows = []
config_bump = LSMCConfig(n_paths=20_000, n_steps_per_year=50, basis_degree=3, seed=42)
pathwise_reference = estimate_delta_pathwise(spot, strike, rate, sigma, maturity, config=config_bump, seed=42)

for bump in [0.10, 0.25, 0.50, 0.75, 1.00]:
    fd_est = estimate_delta_fd(spot, strike, rate, sigma, maturity, config=config_bump, bump=bump, seed=42)
    bump_rows.append({
        "bump": bump,
        "fd_delta": fd_est["estimate"],
        "fd_abs_error": abs(fd_est["estimate"] - benchmark_binomial),
        "pathwise_delta": pathwise_reference["estimate"],
        "pathwise_abs_error": abs(pathwise_reference["estimate"] - benchmark_binomial),
    })

bump_df = pd.DataFrame(bump_rows).round(6)
bump_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

axes[0].plot(bump_df["bump"], bump_df["fd_delta"], marker="^", linewidth=2, label="LSMC finite difference")
axes[0].axhline(pathwise_reference["estimate"], color="#f58518", linestyle="--", linewidth=1.4, label="LSMC pathwise")
axes[0].axhline(benchmark_binomial, color="black", linestyle=":", linewidth=1.2, label="Binomial benchmark")
axes[0].set_title("Delta vs bump size")
axes[0].set_xlabel("Bump size")
axes[0].set_ylabel("Delta")
axes[0].legend()

axes[1].plot(bump_df["bump"], bump_df["fd_abs_error"], marker="^", linewidth=2, label="LSMC finite difference")
axes[1].axhline(abs(pathwise_reference["estimate"] - benchmark_binomial), color="#f58518", linestyle="--", linewidth=1.4, label="LSMC pathwise")
axes[1].set_title("Absolute error vs bump size")
axes[1].set_xlabel("Bump size")
axes[1].set_ylabel("Absolute error vs binomial")
axes[1].legend()

plt.tight_layout()
save_figure(fig, "pathwise_delta_bump_sensitivity.png")
plt.show()


## Robustness 3: Regression Basis Sensitivity

Because the stopping rule is learned by regression, the estimated delta can change when the Laguerre basis changes. This is an observed model-risk check for the current benchmark configuration, not a universal statement about every American option setting.


In [ ]:
basis_rows = []
for basis_degree in [0, 1, 2, 3]:
    config = LSMCConfig(n_paths=20_000, n_steps_per_year=50, basis_degree=basis_degree, seed=42)
    price = lsm_american_put(spot, strike, rate, sigma, maturity, config=config)
    fd_est = estimate_delta_fd(spot, strike, rate, sigma, maturity, config=config, bump=0.5, seed=42)
    pw_est = estimate_delta_pathwise(spot, strike, rate, sigma, maturity, config=config, seed=42)
    basis_rows.append({
        "basis_degree": basis_degree,
        "lsmc_price": price.american_price,
        "fd_delta": fd_est["estimate"],
        "pathwise_delta": pw_est["estimate"],
        "pathwise_se": pw_est["std_error"],
        "fd_abs_error": abs(fd_est["estimate"] - benchmark_binomial),
        "pathwise_abs_error": abs(pw_est["estimate"] - benchmark_binomial),
    })

basis_df = pd.DataFrame(basis_rows).round(6)
basis_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

axes[0].plot(basis_df["basis_degree"], basis_df["lsmc_price"], marker="o", linewidth=2, color="#4c78a8")
axes[0].set_title("Price vs basis degree")
axes[0].set_xlabel("Basis degree")
axes[0].set_ylabel("Price")

axes[1].plot(basis_df["basis_degree"], basis_df["fd_abs_error"], marker="^", linewidth=2, label="LSMC finite difference")
axes[1].plot(basis_df["basis_degree"], basis_df["pathwise_abs_error"], marker="d", linewidth=2, label="LSMC pathwise")
axes[1].set_title("Absolute delta error vs basis degree")
axes[1].set_xlabel("Basis degree")
axes[1].set_ylabel("Absolute error vs binomial")
axes[1].legend()

plt.tight_layout()
save_figure(fig, "pathwise_delta_basis_sensitivity.png")
plt.show()


## Report Interpretation

Taken together, these experiments separate three sources of uncertainty: Monte Carlo sampling variability, bump-size tuning for finite-difference Greeks, and regression-basis risk inside LSMC. The repeated-seed path-count section is especially useful because it strengthens the evidence beyond a single random draw.


## Conclusion

The pathwise delta estimator performs well enough to justify inclusion in the core project narrative: it tracks the external benchmarks reasonably closely, avoids bump-size tuning, and remains competitive in the repeated-seed path-count study. The main caveat remains the same: the LSMC exercise policy is treated as fixed when differentiating, so the results should be presented as validated first-order estimates rather than as a complete treatment of the American exercise-boundary derivative.
